In [18]:
using BootstrapAsymptotics
using QuadGK
using Plots
using StableRNGs: StableRNG
using Statistics
using ProgressBars

In [19]:
κ1    =  2.0 / sqrt(3* pi)
κstar = 0.200364

γ_range = 10 .^ range(-1, stop=2, length=100)
# γ_range = 0.5:0.2:3.0
run_exp = false

sample_over_teacher = 1.0
Δ = 1.0
λ = 1e-4

0.0001

In [20]:
algos = [
    NoResampling(),
    Subsampling(r = 0.8),
    PairBootstrap()
]

# create an empty list for each algorithm
q0_list_dict = Dict()
for algo in algos
    q0_list_dict[algo] = []
end
q1_list_dict = Dict()
for algo in algos
    q1_list_dict[algo] = []
end

exp_q0_list = Dict()
for algo in algos
    exp_q0_list[algo] = []
end
exp_q1_list = Dict()
for algo in algos
    exp_q1_list[algo] = []
end

for γ in ProgressBar(γ_range)
    Δ_add = BootstrapAsymptotics.get_additional_noise_from_kappas(κ1, κstar, γ)
    
    # 
    problem = BootstrapAsymptotics.build_ridge_overparametrized(
    α = sample_over_teacher / γ,
    true_Δ = Δ,
    λ = λ,
    true_ρ = 1.0,
    κ1 = κ1,
    κstar = κstar,
    student_over_teacher_dim = γ,
    )

    for algo in algos
        result = BootstrapAsymptotics.state_evolution(
            problem, algo, algo
        )
        push!(q0_list_dict[algo], result.overlaps.Q[1, 1])
        push!(q1_list_dict[algo], result.overlaps.Q[1, 2])
    end
   # 
   # do not include Δ_add in problem_exp as we generate the data with true Δ
   if !run_exp
       continue
   end

   problem_exp = BootstrapAsymptotics.RidgeOverparametrized(
    α       = sample_over_teacher / γ,
    Δ       = Δ,
    λ       = λ,
    ρ       = 1.0,
    κ1      = κ1,
    κstar   = κstar,
    student_over_teacher_dim = γ
    )

    exp_m, exp_Q = BootstrapAsymptotics.overlaps_empirical(
        StableRNG(0), problem_exp, algo; teacher_dim=500, K=5
    )

    # extract the diagonal and offdiagonal elements of exp_Q
    diag_Q = [exp_Q[i, i] for i in 1:size(exp_Q, 1)]
    offdiag_Q = [exp_Q[i, j] for i in 1:size(exp_Q, 1), j in 1:size(exp_Q, 2) if i != j]
    push!(exp_q0_list_dict, mean(diag_Q))
    push!(exp_q1_list_dict, mean(offdiag_Q))

end

0.0%┣                                              ┫ 0/100 [00:00<00:00, -0s/it]
100.0%┣████████████████████████████████████████┫ 100/100 [00:00<00:00, 5.1kit/s]
100.0%┣████████████████████████████████████████┫ 100/100 [00:00<00:00, 5.1kit/s]


In [21]:
plot(size=(500, 500))
for algo in algos
    plot!(γ_range, q0_list_dict[algo] .- q1_list_dict[algo], label = "$algo")
end
if run_exp
    scatter!(γ_range, exp_q0_list .- exp_q1_list, label = "Empirical variance")
end
plot!(title="Ensemble variance at λ = $λ, n/d = $sample_over_teacher", xscale=:log10, yscale=:log10, xlabel="γ", ylabel="q₀ - q₁")
# save the figure in plots/random_features folder
savefig("plots/random_features/ensemble_variance_λ=$(λ)_n_d=$sample_over_teacher.png")

"/Users/clarte/Code/Uncertainty/Bootstrap/BootstrapAsymptotics/experiments/plots/random_features/ensemble_variance_λ=0.0001_n_d=1.0.png"